# Project 1 — AI vs. Human Text Detection

Course: Intro to AI & Agents, Summer I 2026  
Labels used in the dataset: `0 = Human-written text`, `1 = AI-written text`.

This notebook follows the four required project sections:

1. Data Exploration & Preprocessing
2. Feature Engineering
3. Model Training & Tuning
4. Evaluation & Comparison

The Streamlit app is in `app.py`. The full training implementation is in `train_models.py`.


## 1. Data Exploration & Preprocessing

In this section I load the Excel dataset, keep the `text` and `label` columns, clean empty rows, check the class balance, and tokenize sample text.


In [ ]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = Path('..').resolve()
sys.path.append(str(PROJECT_ROOT))

from project_utils import clean_text, tokenize, compute_linguistic_features

DATA_PATH = PROJECT_ROOT / 'data' / 'training_data' / 'train_data_with_labels.xlsx'
df = pd.read_excel(DATA_PATH)
df = df[['text', 'label']].copy()
df['text'] = df['text'].apply(clean_text)
df = df[df['text'].str.len() > 0].copy()
df['label'] = df['label'].astype(int)

print('Dataset shape:', df.shape)
df['label'].value_counts().sort_index()


In [ ]:
balance = df['label'].value_counts().sort_index()
balance.plot(kind='bar', title='Class Balance')
plt.xticks([0, 1], ['Human (0)', 'AI (1)'], rotation=0)
plt.ylabel('Number of samples')
plt.show()

df['word_count'] = df['text'].apply(lambda x: len(tokenize(x)))
print('Tokenized sample:', tokenize(df.loc[0, 'text'])[:30])
df.groupby('label')['word_count'].describe()


### Preprocessing notes

The cleaning step is simple on purpose. I normalize whitespace and encoding issues, but I keep punctuation because punctuation patterns, sentence structure, and readability can be useful signals for AI vs. human writing.


## 2. Feature Engineering

I implemented and compared the three feature groups required in the project:

- **TF-IDF:** word and phrase frequency features.
- **Word2Vec average embeddings:** word vectors trained from the dataset.
- **Linguistic features:** word count, sentence count, average sentence length, vocabulary richness, punctuation, uppercase ratio, digit ratio, and readability.

The final Streamlit app uses TF-IDF plus linguistic features for the traditional ML models because this gave the best balance between performance and explanation.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from scipy.sparse import csr_matrix, hstack

X_train_text, X_test_text, y_train, y_test = train_test_split(
    df['text'].values,
    df['label'].values,
    test_size=0.2,
    stratify=df['label'].values,
    random_state=42,
)

vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    stop_words='english',
    sublinear_tf=True,
)
X_train_tfidf = vectorizer.fit_transform(X_train_text)
X_test_tfidf = vectorizer.transform(X_test_text)

scaler = StandardScaler()
X_train_ling = scaler.fit_transform(compute_linguistic_features(X_train_text))
X_test_ling = scaler.transform(compute_linguistic_features(X_test_text))

X_train_combined = hstack([X_train_tfidf, csr_matrix(X_train_ling)], format='csr')
X_test_combined = hstack([X_test_tfidf, csr_matrix(X_test_ling)], format='csr')

print('Combined feature shape:', X_train_combined.shape)


### Feature comparison results

| feature_set | accuracy | precision | recall | f1 | roc_auc |
|---|---:|---:|---:|---:|---:|
| TF-IDF | 0.930 | 0.909 | 0.956 | 0.932 | 0.983 |
| Linguistic Features | 0.722 | 0.696 | 0.788 | 0.739 | 0.772 |
| Word2Vec Average Embeddings | 0.610 | 0.608 | 0.620 | 0.614 | 0.646 |

TF-IDF performed best in the saved run. Linguistic features were weaker alone, but they are useful for explanation. Word2Vec was included to compare an embedding-based representation against the frequency-based features.


In [ ]:
feature_results = pd.read_csv(PROJECT_ROOT / 'reports' / 'feature_comparison.csv')
feature_results


## 3. Model Training & Tuning

I trained six classifiers: SVM, Decision Tree, AdaBoost, FNN, LSTM, and CNN for text.

The full training code is implemented in `train_models.py` to keep the notebook readable. The notebook loads the saved training outputs and documents the tuning setup below.

For the traditional machine learning models, I used `GridSearchCV` with F1-score as the selection metric. For the deep learning models, I used a small manual grid because FNN, LSTM, and CNN take longer to train on a normal laptop.


In [ ]:
# Full training command used for the saved model artifacts:
# !python ../train_models.py

# Faster testing run for debugging:
# !python ../train_models.py --fast

ml_parameter_grids = {
    'SVM': {'C': [0.5, 1.0, 2.0]},
    'Decision Tree': {'max_depth': [10, 20, None], 'min_samples_split': [2, 10]},
    'AdaBoost': {'n_estimators': [50, 100], 'learning_rate': [0.5, 1.0]},
}

deep_learning_tuning = {
    'FNN': ['embedding_dim', 'hidden_dim', 'dropout', 'learning_rate', 'epochs'],
    'LSTM': ['max_len', 'embedding_dim', 'hidden_dim', 'dropout', 'learning_rate', 'epochs'],
    'CNN': ['max_len', 'embedding_dim', 'num_filters', 'filter_sizes', 'dropout', 'learning_rate', 'epochs'],
}

ml_parameter_grids, deep_learning_tuning


In [ ]:
model_results = pd.read_csv(PROJECT_ROOT / 'reports' / 'model_comparison.csv')
tuning_plan = pd.read_csv(PROJECT_ROOT / 'reports' / 'tuning_plan.csv')
deep_tuning = pd.read_csv(PROJECT_ROOT / 'reports' / 'deep_tuning_results.csv')

print('Model comparison:')
display(model_results[['model', 'accuracy', 'precision', 'recall', 'f1', 'roc_auc', 'best_params']])

print('Tuning plan:')
display(tuning_plan)

print('Selected deep learning configurations:')
display(deep_tuning)


### Tuning notes

The traditional ML models used `GridSearchCV`. The deep learning models did not stay at default settings; I tested practical values for sequence length, embedding size, hidden units or filters, dropout, learning rate, and epochs. The final saved models are loaded by `app.py` in the Streamlit application.


## 4. Evaluation & Comparison

I evaluated every model with accuracy, precision, recall, F1-score, confusion matrix, ROC curve, and ROC-AUC. I focused mostly on F1-score because it balances precision and recall.


### Model comparison results

| model | accuracy | precision | recall | f1 | roc_auc |
|---|---:|---:|---:|---:|---:|
| SVM | 0.938 | 0.936 | 0.940 | 0.938 | 0.989 |
| AdaBoost | 0.914 | 0.888 | 0.948 | 0.917 | 0.976 |
| FNN | 0.876 | 0.870 | 0.884 | 0.877 | 0.952 |
| CNN | 0.866 | 0.865 | 0.868 | 0.866 | 0.942 |
| Decision Tree | 0.844 | 0.812 | 0.896 | 0.852 | 0.836 |
| LSTM | 0.756 | 0.750 | 0.768 | 0.759 | 0.792 |


In [ ]:
plot_df = model_results.sort_values('f1', ascending=True)
plt.figure(figsize=(8, 4))
plt.barh(plot_df['model'], plot_df['f1'])
plt.xlabel('F1-score')
plt.title('Model F1-score Comparison')
plt.xlim(0, 1)
plt.show()


### ROC curve comparison

![ROC model comparison](../reports/figures/roc_model_comparison.png)

### Confusion matrices

SVM:

![SVM confusion matrix](../reports/figures/confusion_svm.png)

Decision Tree:

![Decision Tree confusion matrix](../reports/figures/confusion_decision_tree.png)

AdaBoost:

![AdaBoost confusion matrix](../reports/figures/confusion_adaboost.png)

FNN:

![FNN confusion matrix](../reports/figures/confusion_fnn.png)

LSTM:

![LSTM confusion matrix](../reports/figures/confusion_lstm.png)

CNN:

![CNN confusion matrix](../reports/figures/confusion_cnn.png)


### Required evaluation checklist

- Accuracy, precision, recall, F1-score, and ROC-AUC are shown for all six models.
- Confusion matrices are included for all six models.
- ROC curves are saved in the reports folder, including one combined ROC comparison plot.
- The written analysis below explains the best model, feature impact, model weaknesses, and real-world limitations.

### Written analysis

The best model in my saved run was **SVM** with an F1-score of **0.938**. SVM performed strongly because TF-IDF creates a high-dimensional text representation, and SVM usually works well with that type of data.

The linguistic features by themselves were weaker than TF-IDF, but they still helped explain the text style. Features like sentence length, vocabulary richness, punctuation ratio, and readability gave extra signals that are easier to understand than word weights alone.

The deep learning models worked, but they were more sensitive to training time and sequence length. CNN and FNN performed better than LSTM in this saved run. My main takeaway is that simpler ML models can be very strong for this type of text classification, especially when the dataset is not huge.

I would not use this kind of tool as the only decision-maker in a classroom. It can be useful as a warning signal, but AI detection is not perfect. A human review would still be needed before making any serious academic decision.
